# Rouge & BertScore

In [1]:
# Rouge Score
from rouge import Rouge 

def eval_rouge(hypothesis, reference):
    rouge = Rouge(stats=["f"])
    scores = rouge.get_scores(hypothesis, reference)
    
    print("F1 Scores: ", scores)
    
    return scores

In [2]:
from bert_score import score
from transformers import AutoTokenizer, AutoModel

def eval_bertScore(output_summary_list, benchmark_summary_list):
    # 원하는 모델 로드
    model_type = "microsoft/deberta-xlarge-mnli"
    
    # BERTScore 함수에서 모델 타입을 직접 설정
    P, R, F1 = score(output_summary_list, benchmark_summary_list, lang="en", model_type=model_type, verbose=True)
    
    # bertScore = F1.mean().item()
    # F1 리스트를 파이썬 리스트로 변환
    f1_scores = F1.tolist()
    
    # F1 점수들의 합계를 계산
    f1_sum = sum(f1_scores)
    print("bertScore - F1 Score:", f1_sum)

    return f1_sum

# CASPR

In [ ]:
import pandas as pd
from transformers import pipeline
import numpy as np

def decompose_sentences(text):
    # 실제로는 GPT-3.5를 사용하여 복잡한 문장을 단일 주장 문장으로 분해합니다.
    # 여기서는 간단히 문장을 분할합니다.
    sentences = text.split('.')
    return sentences

# NLI 모델 로드
nli_model = pipeline("text-classification", model="roberta-large-mnli", device=0)

# 문장 간 논리적 관계 평가
def nli_score(sentence1, sentence2):
    result1 = nli_model(f"{sentence1.strip()} entails {sentence2.strip()}")
    result2 = nli_model(f"{sentence2.strip()} entails {sentence1.strip()}\n")
    
    label1 = result1[0]['label']
    label2 = result2[0]['label']
    
    # 핵심 정보만 출력
    print(f"Comparing: '{sentence1}' <-> '{sentence2}'")
    print(f" -> Result 1: {label1}, Result 2: {label2}")
    
    if label1 == 'CONTRADICTION' and label2 == 'ENTAILMENT':
        return 'neutral'
    elif label1 == 'ENTAILMENT' and label2 == 'CONTRADICTION':
        return 'neutral'
    elif label1 == 'CONTRADICTION' or label2 == 'CONTRADICTION':
        return 'contradiction'
    elif label1 == 'ENTAILMENT' or label2 == 'ENTAILMENT':
        return 'entailment'
    else:
        return 'neutral'

# CASPR 점수 계산
def calculate_caspr(summary1, summary2):
    flag = True     # 첫 번째가 summary1, 두 번째가 summary2
    scores = []
    
    for i in range(0, 2):
        if flag:
            sentences1 = decompose_sentences(summary1)
            sentences2 = decompose_sentences(summary2)
        else:
            sentences1 = decompose_sentences(summary2)
            sentences2 = decompose_sentences(summary1)
        
        for s1 in sentences1:
            contradictions = 0
            entailments = 0 
            neutrals = 0
            
            if not s1.strip():
                continue
            
            for s2 in sentences2:
                if not s2.strip():
                    continue
                
                relationship = nli_score(s1, s2)
                if relationship == 'contradiction':
                    contradictions += 1
                elif relationship == 'entailment':
                    entailments += 1
                else:
                    neutrals += 1
                    
            # 개수 세서 점수 매기기 
            if contradictions == 0 and entailments == 0: # neutrals == len(sentences2)
                # 모든 문장들이 다 Neutral이므로, +1
                scores.append(1)
            elif contradictions > entailments:
                scores.append(1)
            else:
                scores.append(-1)
           
        flag = False # 두 번째 반복에서는 summary2와 summary1을 바꿈
        
    if not scores:
        return 0
    
    # 핵심 정보만 출력
    print(f"Calculated Scores: {scores}")    
    
    caspr_score = (sum(scores) / len(scores) + 1) * 50  # Adjusted to range [0, 100]
    return caspr_score

# aspect 유사도 - GPT 임베딩 + cosine similarity

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
SERVICE_KEY = os.getenv('OPEN_AI_KEY')

print(SERVICE_KEY)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

def get_gpt_embedding(text):
      client = OpenAI(api_key=SERVICE_KEY)
      response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"  # GPT embedding model
    )
      embedding = response.data[0].embedding

      return embedding

def aspect_similar(bench_aspect, output_aspect):   
   # Get embeddings for both aspects
   bench_embedding = get_gpt_embedding(bench_aspect)
   output_embedding = get_gpt_embedding(output_aspect)

   # Compute cosine similarity between the embeddings
   similarity_score = cosine_similarity([bench_embedding], [output_embedding])[0][0]
   
   return similarity_score

# 성능 평가

## 유사성 평가 (벤치마크와 최종요약)

In [6]:
import pandas as pd
import sys
import warnings

# 경고를 출력하는 함수
def warning_handler(message, category, filename, lineno, file=None, line=None):
    print(f"[WARNING] {message} in {filename} at line {lineno}")

# 경고 처리기 설정 (경고는 출력하되, 프로그램 종료는 하지 않음)
warnings.showwarning = warning_handler

# CASPR 점수 계산을 위한 함수 (output_json의 common_aspects만 사용)
def calculate_cont_aspects_caspr(output_json, caspr_total, caspr_count):    
    # output_json에서 common_aspects를 가져옴
    common_aspects_output = output_json.get('common_aspects', {})
    
    # common_aspects의 각 항목에서 cont 부분을 처리
    for aspect, aspect_data in common_aspects_output.items():
        cont_data = aspect_data.get('cont', {})
        
        # itemA와 itemB가 모두 있어야 CASPR 계산
        if 'itemA' in cont_data and 'itemB' in cont_data:
            itemA_text = cont_data['itemA']
            itemB_text = cont_data['itemB']
            
            print(f"Processing CASPR for aspect '{aspect}':")
            print(f"  - itemA: {itemA_text}")
            print(f"  - itemB: {itemB_text}")
            
            # CASPR 점수 계산 (itemA와 itemB의 문장 비교)
            caspr_score = calculate_caspr(itemA_text, itemB_text)
            caspr_total += caspr_score
            caspr_count += 1
        else:
            print(f"[INFO] Aspect '{aspect}' is missing 'itemA' or 'itemB'. Skipping CASPR calculation for this aspect.")
    
    return caspr_total, caspr_count

# 유사도 평가를 위한 함수
def MaxAspectSimilarEval(output_json, benchmark_json, number): 
    rouge_1 = 0
    rouge_L = 0
    rouge_1_wo_asp = 0
    rouge_L_wo_asp = 0
    aspect_correctness_matches = 0  # 정확하게 맞춘 aspect 개수
    aspect_correctness_total = 0    # 평가된 aspect의 총 개수
    caspr_total = 0
    caspr_count = 0
    bert_benchList = []
    bert_outputList = []
    
    print("="* 50)
    print(f"{number}번째 파일 처리 중 ... \n")
    print("="* 50)
    
    ###############################################################################################################################

    # CASPR 점수 계산 (common_aspects의 cont에서 itemA와 itemB의 텍스트 사용)
    caspr_total, caspr_count = calculate_cont_aspects_caspr(output_json, caspr_total, caspr_count)

    ###############################################################################################################################


    # common_aspects 처리
    try:
        common_aspects_bench = benchmark_json.get('common_aspects', {})
        common_aspects_output = output_json.get('common_aspects', {})
    except Exception as e:
        print(f"[ERROR] Failed to process common_aspects: {e}")
        sys.exit(1)

    print(f"Benchmark common aspects: {list(common_aspects_bench.keys())}")
    print(f"Output common aspects: {list(common_aspects_output.keys())}")
        
    # 벤치마크와 모델에서 유사한 aspect를 찾고 평가
    
    if common_aspects_bench :
        try:
            # 변경된 함수 호출 (process_aspects_common 함수로 comm 및 cont 처리)
            rouge_1, rouge_L, aspect_correctness_matches, aspect_correctness_total = process_aspects_common(
                common_aspects_bench, common_aspects_output, bert_benchList, bert_outputList, 
                rouge_1, rouge_L, aspect_correctness_matches, aspect_correctness_total
            )
            
        except Exception as e:
            print(f"[ERROR] Failed to process common aspects: {e}")
            sys.exit(1)
    else : 
        print(f"\n benchmark에 common aspect가 존재하지 않음 !!!")

    ###############################################################################################################################
    
    # unique_aspects 처리
    try:
        unique_aspects_bench = benchmark_json.get('unique_aspects', {})
        unique_aspects_output = output_json.get('unique_aspects', {})
    except Exception as e:
        print(f"[ERROR] Failed to process unique_aspects: {e}")
        sys.exit(1)

    print(f"Unique aspects for itemA and itemB:")
    
    try:
        for item_name in unique_aspects_bench:
            if not unique_aspects_bench[item_name]:
                print(f"[INFO] No unique aspects to evaluate for file: '{item_name}'\n")
                continue

            # unique_aspects_output에 대응하는 item이 있는지 확인
            if item_name in unique_aspects_output:
                print(f"\nProcessing  unique aspect {item_name}:")
                bench_aspects = unique_aspects_bench[item_name]
                output_aspects = unique_aspects_output[item_name]

                # 각 benchmark aspect를 output의 모든 aspect와 비교하여 가장 유사한 aspect를 찾음
                for aspect, bench_value in bench_aspects.items():
                    max_similarity = 0
                    best_output_aspect = None
                    best_output_value = None

                    # output의 각 aspect와 비교
                    for output_aspect, output_value in output_aspects.items():
                        try:
                            similarity = aspect_similar(aspect, output_aspect)  # 유사도 계산 함수
                            print(f"Similarity between '{aspect}' and '{output_aspect}': {similarity}")

                            if similarity > max_similarity:
                                max_similarity = similarity
                                best_output_aspect = output_aspect
                                best_output_value = output_value
                        except Exception as e:
                            print(f"[ERROR] Failed to calculate similarity between '{aspect}' and '{output_aspect}': {e}")
                            continue
                    
                    if best_output_aspect:  
                        if (max_similarity >= 0.4):     # 0.4 이상인 aspect들끼리만 비교 
                            # 가장 유사한 output aspect와 평가
                            print(f"\n 0.4  > 이상 < 인 aspect들 끼리만 비교 수행  -------------- ")
                            print(f"\nMost similar aspect for '{aspect}' is '{best_output_aspect}' with similarity: {max_similarity}")
                            print(f"[Benchmark - {item_name}] Aspect: '{aspect}', Value: '{bench_value}'")
                            print(f"[Output - {item_name}] Aspect: '{best_output_aspect}', Value: '{best_output_value}'")

                            # BERT 및 Rouge 비교 
                            rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                                bench_value, best_output_value, rouge_1, rouge_L, bert_benchList, bert_outputList
                            )
                            
                            # Aspect Accuracy 계산
                            aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                                bench_value, best_output_value, aspect_correctness_matches, aspect_correctness_total
                            )

                            print(f"Rouge and BERTScore calculated for aspect '{aspect}' between Benchmark and Output.\n")
                                
                        # 0.4 미만인 aspect들은 0점 처리                    
                        else :  
                            print(f"\n 0.4 > 미만 <인 aspect들 끼리만 비교 수행  ________________ ")
                            print(f"\nMost similar aspect for '{aspect}' is '{best_output_aspect}' with similarity: {max_similarity}")
                            print(f"[Benchmark - {item_name}] Aspect: '{aspect}', Value: '{bench_value}'")
                            print(f"[Output - {item_name}] Aspect: '{best_output_aspect}', Value: '{best_output_value}'")

                            # BERT 및 Rouge 비교 
                            rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                                "", "", rouge_1, rouge_L, bert_benchList, bert_outputList
                            )
                            
                            # Aspect Accuracy 계산 
                            aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                                bench_value, "", aspect_correctness_matches, aspect_correctness_total
                            )

                            print(f"Rouge and BERTScore calculated for aspect '{aspect}' between Benchmark and Output.\n")
                    else:
                        # 해당하는 output aspect가 없을 경우 0점 처리
                        print(f"[WARNING] No similar aspect found for '{aspect}' in output for '{item_name}'. Setting scores to 0.")
                        rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                            bench_value, "", rouge_1, rouge_L, bert_benchList, bert_outputList
                        )
                        # Aspect Accuracy 계산 
                        aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                            bench_value, "", aspect_correctness_matches, aspect_correctness_total
                        )

            else: # uniq_aspect에 대응되는 item_name이 없으면, 0점 처리해야함
                print(f"[WARNING] No data for '{item_name}' in output.")
                rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                bench_value, "", rouge_1, rouge_L, bert_benchList, bert_outputList
                )
                
                # Aspect Accuracy 계산 
                aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                    bench_value, "", aspect_correctness_matches, aspect_correctness_total
                )

            
    except Exception as e:
        print(f"[ERROR] Failed to process unique aspects: {e}")
        sys.exit(1)    
    
    
    # 점수 평균화
    total_aspects = len(bert_benchList)
    print(f'\nTotal aspects evaluated: {total_aspects}')
    
    if total_aspects <= 0:
        print("[ERROR] No aspects to evaluate. Check the input data.")
        sys.exit(1)
    
    # BERTScore 계산
    if bert_outputList and bert_benchList:
        print(f"\n ----> TOTAL - Bert_Bench_List : {bert_benchList}")
        print(f"\n ----> TOTAL - Bert_Our_Output_List : {bert_outputList}")
        print("\n\n[INFO] Calculating BERTScore...")
        try:
            bertScore = eval_bertScore(bert_outputList, bert_benchList)
        except Exception as e:
            print(f"[ERROR] Failed to calculate BERTScore: {e}")
            sys.exit(1)
    else:
        print("[ERROR] BERTScore cannot be calculated due to empty lists.")
        sys.exit(1)
        
    rouge_1_wo_asp, rouge_L_wo_asp, BS_wo_asp = without_aspect_score (output_json, benchmark_json)
    print(f"\n\nwithout_aspect - only total text score ::\n rouge-1 : {rouge_1_wo_asp}, rouge-L : {rouge_L_wo_asp}, BertScore : {BS_wo_asp}")
    
    print(f"\n\nAspect Correctness Matches: {aspect_correctness_matches }%")
    
    
    return total_aspects, caspr_count, aspect_correctness_total, rouge_1, rouge_L, rouge_1_wo_asp, rouge_L_wo_asp, bertScore, BS_wo_asp, caspr_total, aspect_correctness_matches


# 새로운 함수: Aspect Accuracy 계산
def calculate_aspect_accuracy(benchmark_text, output_text, correctness_matches, correctness_total):
    """
    참조 요약(benchmark_text)과 모델 생성 요약(output_text)을 비교하여
    Accuracy를 평가하는 함수.
    """

    # benchmark_text와 output_text 모두 있는 경우
    if benchmark_text and output_text:
        correctness_matches += 1  # 정확히 맞춘 것으로 간주
        correctness_total += 1  # 평가된 aspect 개수 증가

    # benchmark_text는 있는데 output_text가 없는 경우
    elif benchmark_text and not output_text:
        correctness_total += 1  # 정확히 맞추지 못한 것으로 간주

    # benchmark_text는 없는데 output_text가 있는 경우
    elif not benchmark_text and output_text:
        correctness_total += 1  # 정확히 맞추지 못한 것으로 간주

    # 둘 다 없는 경우는 평가하지 않고 건너뜀
    # else 블록으로 처리하지 않음
    
    return correctness_matches, correctness_total


# aspect 따지지 않고, 텍스트를 하나로 모아서 rouge, bs 계산하는 함수
def without_aspect_score(output_json, benchmark_json):
    
    r1_socre = 0
    rl_score = 0
    bs_score = 0
    
    opt_text = extract_and_concatenate_text(output_json)
    ben_text = extract_and_concatenate_text(benchmark_json)
    
    rougeScore = eval_rouge(opt_text, ben_text)[0]
    r1_socre = rougeScore["rouge-1"]["f"]
    rl_score = rougeScore["rouge-l"]["f"]
    
    bs_score = eval_bertScore([opt_text], [ben_text])
    
    return r1_socre, rl_score, bs_score
    
def ensure_period(text):
    """텍스트가 마침표로 끝나지 않으면 마침표 추가"""
    if not text.endswith('.'):
        return text + '.'
    return text

def extract_and_concatenate_text(data):
    result_text = []
    
    # 재귀적으로 데이터를 탐색하여 최하위 value들만 추출
    def recursive_extract(obj):
        if isinstance(obj, dict):
            for value in obj.values():
                recursive_extract(value)
        elif isinstance(obj, str):
            # 마침표가 없으면 추가
            result_text.append(ensure_period(obj))

    recursive_extract(data)
    return " ".join(result_text)


# 공통 aspects 처리 함수 
def process_aspects_common(benchmark_aspects, output_aspects, bert_benchList, bert_outputList, rouge_1, rouge_L, aspect_correctness_matches, aspect_correctness_total):
    for ben_aspect, ben_data in benchmark_aspects.items():
        max_similarity = 0
        max_output_aspect = None
    
        print(f"\nCommon aspect: '{ben_aspect}' 처리 중 ... ")
        
        # output apsect가 존재하는 경우
        if output_aspects:
            # 모델 요약에서 가장 유사한 aspect 찾기
            for output_aspect, output_data in output_aspects.items():
                try:
                    similarity = aspect_similar(ben_aspect, output_aspect) 
                except Exception as e:
                    print(f"[ERROR] Failed to calculate similarity: {e}")
                    sys.exit(1)

                print(f" - Similarity between '{ben_aspect}' and '{output_aspect}': {similarity}")
                if similarity > max_similarity:
                    max_similarity = similarity
                    max_output_aspect = output_aspect
            
            # aspect similarity가 0.4 이상인 경우
            if (max_similarity >= 0.4):
                # 가장 유사한 aspect 출력
                print(f"\n 0.4  > 이상 < 인 aspect들 끼리만 비교 수행  -------------- ")
                print(f"\nMost similar aspect for '{ben_aspect}' is '{max_output_aspect}' with similarity: {max_similarity}")
                
                # 해당하는 output aspect가 없을 경우 0점 처리
                if max_output_aspect is None:
                    print(f"[WARNING] No similar output aspect found for '{ben_aspect}'. Setting scores to 0.")
                    rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                            "", "", rouge_1, rouge_L, bert_benchList, bert_outputList
                        )
                    
                    # Aspect Accuracy 계산 (common_aspects) - 0점 처리 필요
                    aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                        ben_data['comm'], "", aspect_correctness_matches, aspect_correctness_total
                    )
                    
                    continue  # 다음 aspect로 넘어감
                
                ########################## common-aspects에 대해서 COMM 부분 처리 #############################
                
                benchmark_comm = ben_data['comm']  
                output_comm = output_aspects[max_output_aspect]['comm']
                
                # Aspect Accuracy 계산 (common_aspects) 
                aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                    benchmark_comm, output_comm, aspect_correctness_matches, aspect_correctness_total
                )

                # comm 문장 비교 
                if 'comm' in ben_data and 'comm' in output_aspects[max_output_aspect]:
                    print(f"\nbenchmark와 our output에 comm이 둘 다 있는 경우  ----------------- ")
                    
                    benchmark_comm = ben_data['comm']  
                    output_comm = output_aspects[max_output_aspect]['comm']
                   
                    # Rouge 및 BERT 계산
                    rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                        benchmark_comm, output_comm, rouge_1, rouge_L, bert_benchList, bert_outputList
                    )
                    
                    
                # ben_data에만 comm이 있을 경우 '0점 처리'
                elif 'comm' in ben_data and 'comm' not in output_aspects[max_output_aspect]:
                    print(f"\nbenchmark에만 comm이 있는 경우 (0점 처리) ----------------- ")

                    benchmark_comm = ben_data['comm']
                    output_comm = ""  # output에서 comm이 없는 경우 처리 방식
                    
                    # 필요한 처리 또는 비교
                    rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                        benchmark_comm, output_comm, rouge_1, rouge_L, bert_benchList, bert_outputList
                    )

                # output_aspects에만 comm이 있을 경우 benchmark에는 없으므로, '건너뜀'
                elif 'comm' not in ben_data and 'comm' in output_aspects[max_output_aspect]:
                    print(f"\nour output에만 comm이 있는 경우 (벤치마크에 없으므로 패스) ----------------- ")
                    pass

                # 둘 다 comm이 없을 경우 처리 (필요 시)
                else:
                    print(f"\nbenchmark와 our output에 comm이 둘 다 없는 경우 (패스)  ----------------- ")
                    pass
                
                ########################## common-aspects에 대해서 CONT 부분 정확도 처리 (CASPR은 다른 함수로 구현) #############################
                
                # cont 부분에서 itemA와 itemB를 비교하여 CASPR 계산 
                if 'cont' in output_aspects[max_output_aspect]:
                    benchmark_cont = ben_data.get('cont', {})
                    output_cont = output_aspects[max_output_aspect].get('cont', {})
                    
                    for item_n in ['itemA', 'itemB']:
                        benchmark_cont_text = benchmark_cont.get(item_n, "")
                        output_cont_text = output_cont.get(item_n, "")
                        
                        # Aspect Accuracy 계산 (common_aspects) 
                        aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                            benchmark_cont_text, output_cont_text, aspect_correctness_matches, aspect_correctness_total
                        )
            
            # 0.4 미만인 aspect들은 0점 처리                    
            else :  
                print(f"\n 0.4 > 미만 <인 aspect들 끼리만 비교 수행  ________________ ")
                print(f"\nMost similar aspect for '{ben_aspect}' is '{max_output_aspect}' with similarity: {max_similarity}")
                print(f"[Benchmark] Aspect: '{ben_aspect}', Value: '{ben_data}'")
                print(f"[Output] Aspect: '{max_output_aspect}', Value: '{output_aspects[max_output_aspect]}'")

                # BERT 및 Rouge 비교 
                rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                    "", "", rouge_1, rouge_L, bert_benchList, bert_outputList
                )
                
                # Aspect Accuracy 계산 (common_aspects) 
                aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                    ben_data['comm'], "", aspect_correctness_matches, aspect_correctness_total
                )

                print(f"Rouge and BERTScore calculated for aspect '{ben_aspect}' between Benchmark and Output.\n")
                
              
        #  output aspect가 존재하지 않는 경우       
        else:
            print(f"\noutput에 aspect 존재하지 않음 --> 0점 처리 \n")
            # Rouge 및 BERT 계산 
            rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                "", "", rouge_1, rouge_L, bert_benchList, bert_outputList
            )
            aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                ben_data['comm'], "", aspect_correctness_matches, aspect_correctness_total
            )

    return rouge_1, rouge_L, aspect_correctness_matches, aspect_correctness_total


# unique_aspects 처리 함수
def process_aspects_unique(benchmark_aspects, output_aspects, bert_benchList, bert_outputList, rouge_1, rouge_L, item_name):
    for ben_aspect, ben_data in benchmark_aspects.items():
        max_similarity = 0
        max_output_aspect = None
        
        print(f"\n[UNIQUE aspect] {ben_aspect} from Benchmark aspect 처리 중 ..... \n")
        
        # 모델 요약에서 가장 유사한 aspect 찾기
        for output_aspect, output_data in output_aspects.items():
            try:
                similarity = aspect_similar(ben_aspect, output_aspect)
            except Exception as e:
                print(f"[ERROR] Failed to calculate similarity: {e}")
                sys.exit(1)

            print(f" - Similarity between '{ben_aspect}' and '{output_aspect}': {similarity}")
            if similarity > max_similarity:
                max_similarity = similarity
                max_output_aspect = output_aspect

        # 가장 유사한 aspect 출력
        print(f"Most similar aspect for '{ben_aspect}' is '{max_output_aspect}' with similarity: {max_similarity}")

        if max_output_aspect is None:
            print(f"[WARNING] No similar output aspect found for '{ben_aspect}'. Setting scores to 0.")
            # BERT 및 Rouge 비교 
            rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
                ben_data, "", rouge_1, rouge_L, bert_benchList, bert_outputList
            )
            
            aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
                ben_data, "", aspect_correctness_matches, aspect_correctness_total
            )
            continue  # 다음 aspect로 넘어감
        

        # BERT 및 Rouge 비교 
        rouge_1, rouge_L, bert_benchList, bert_outputList = calculate_rouge_and_bert(
            ben_data, output_aspects[max_output_aspect], rouge_1, rouge_L, bert_benchList, bert_outputList
        )
        
        aspect_correctness_matches, aspect_correctness_total = calculate_aspect_accuracy(
            ben_data, output_aspects[max_output_aspect], aspect_correctness_matches, aspect_correctness_total
        )

    return rouge_1, rouge_L


# Rouge와 BERT 평가 계산 함수 
def calculate_rouge_and_bert(benchmark_text, output_text, rouge_1, rouge_L, bert_benchList, bert_outputList):
    if (benchmark_text == "" or output_text == ""):
        bert_benchList.append(benchmark_text)
        bert_outputList.append(output_text)
        
        return rouge_1, rouge_L, bert_benchList, bert_outputList
    
    else:
        # BERT 평가 리스트에 추가
        bert_benchList.append(benchmark_text)
        bert_outputList.append(output_text)
        
        print(f"[INFO] Rouge and BERTScore are being calculated for:")
        print(f"  - Benchmark text: {benchmark_text}")
        print(f"  - Output text: {output_text}\n")

        # Rouge 점수 계산
        rougeScore = eval_rouge(output_text, benchmark_text)[0]
        rouge_1 += rougeScore["rouge-1"]["f"]
        rouge_L += rougeScore["rouge-l"]["f"]
        
        return rouge_1, rouge_L, bert_benchList, bert_outputList

### CoCoTrip 평가 돌리는 코드

In [ ]:
import os
import json
import pandas as pd


# 폴더 경로 지정
result_folder = '/workspace/COLING2025_new/STRUM-LLM/Experiment_result/coco_47'
bench_folder = '/workspace/COLING2025_new/benchmark/eval-cocotrip-final-bench-ffffffffff' # 고정 

# MaxAspectSimilarEval 함수를 처리하는 함수
def process_files(result_folder, bench_folder, folder_name):
    result_path = os.path.join(result_folder, folder_name)
    bench_path = os.path.join(bench_folder, folder_name)
    
    # 비교 횟수
    num_of_compare = 0
    num_CASPR = 0
    num_aspect_accuracy = 0
    
    # 총 합
    total_rouge_1_aspect = 0
    total_rouge_L_aspect = 0
    total_rouge_1_without_aspect = 0
    total_rouge_L_without_aspect = 0
    total_BertScore_aspect = 0
    total_BertScore_without_aspect = 0
    total_CASPR = 0
    total_aspect_Accuracy = 0
    
    # result 폴더의 파일들 순회
    for result_file in os.listdir(result_path):
        
        # 비교 횟수
        num_of_comp = 0
        num_CA = 0
        num_aspect_acc = 0
        
        # 총 합
        total_rouge_1_asp = 0
        total_rouge_L_asp = 0
        total_rouge_1_without_asp = 0
        total_rouge_L_without_asp = 0
        total_BertScore_asp = 0
        total_BertScore_without_asp = 0
        total_CA = 0
        total_aspect_Acc = 0
        
        if result_file.split('_')[0].isdigit() and result_file.endswith('_ours_output.json'):
            # 숫자 추출 (예: 'aspect_review[0]_new.json'에서 '0' 추출)
            number = result_file.split('[')[1].split(']')[0]
            
            # 대응하는 bench 파일도 동일한 형식으로 설정
            bench_file = f'aspect_review[{number}]_new.json'
            result_file_path = os.path.join(result_path, result_file)
            bench_file_path = os.path.join(bench_path, bench_file)
            
            print(f'result_file_path : {result_file_path}\n')
            print(f'bench_file_path : {bench_file_path}\n')

            
            # 두 파일 모두 존재할 경우 함수 실행
            if os.path.exists(result_file_path) and os.path.exists(bench_file_path):
                # JSON 파일 읽기
                with open(result_file_path, 'r', encoding='utf-8') as rf:
                    output_json = json.load(rf)
                
                with open(bench_file_path, 'r', encoding='utf-8') as bf:
                    benchmark_json = json.load(bf)
                
                # MaxAspectSimilarEval 함수 실행
                num_of_comp, num_CA, num_aspect_acc, total_rouge_1_asp, total_rouge_L_asp, total_rouge_1_without_asp, total_rouge_L_without_asp, total_BertScore_asp, total_BertScore_without_asp, total_CA, total_aspect_Acc  = MaxAspectSimilarEval(output_json, benchmark_json, number)
                
                # 비교 횟수
                num_of_compare += num_of_comp
                num_CASPR += num_CA
                num_aspect_accuracy += num_aspect_acc
                
                # 총 합
                total_rouge_1_aspect += total_rouge_1_asp
                total_rouge_L_aspect += total_rouge_L_asp
                total_rouge_1_without_aspect += total_rouge_1_without_asp
                total_rouge_L_without_aspect += total_rouge_L_without_asp
                total_BertScore_aspect += total_BertScore_asp
                total_BertScore_without_aspect += total_BertScore_without_asp
                total_CASPR += total_CA
                total_aspect_Accuracy += total_aspect_Acc
                
                
                print(f'Processed {folder_name} - {result_file} with number {number}')
            else:
                print(f'Missing files for number {number} in {folder_name}')
                
    
    # 결과 데이터를 DataFrame으로 저장
    data = {
        "rouge_1_aspect": [total_rouge_1_aspect/num_of_compare],
        "rouge_L_aspect": [total_rouge_L_aspect/num_of_compare],
        "rouge_1_without_aspect": [total_rouge_1_without_aspect/18],
        "rouge_L_without_aspect": [total_rouge_L_without_aspect/18],
        "bertScore_aspect": [total_BertScore_aspect/num_of_compare],
        "bertScore_without_aspect": [total_BertScore_without_aspect/18],
        "CASPR": [total_CASPR/num_CASPR],
        "aspect Accuracy" : [total_aspect_Accuracy/num_aspect_accuracy]
    }
    
    # pandas DataFrame 생성
    df = pd.DataFrame(data)
    
    # 결과 DataFrame 반환
    return df

folders = ['test']

for folder in folders:
    final_df = process_files(result_folder, bench_folder, folder)

    
final_df.to_csv(f'{result_folder}/final_result_avaerage.csv', index=False)